# TCN (red convolucional temporal): proceso de construcción

Cuaderno de proceso del modelo convolucional. Las celdas de código son lecturas ligeras (no
entrenan nada) y avisan si falta algún archivo. Modelos hermanos:
[`01_xgboost_booster.ipynb`](01_xgboost_booster.ipynb) y
[`03_cnn_lstm.ipynb`](03_cnn_lstm.ipynb).

**El TCN es solo convolucional**: no tiene ninguna parte recurrente. Es la referencia para medir
qué aporta añadir una LSTM (el híbrido CNN-LSTM del tercer cuaderno).

In [ ]:
import os
import warnings
from datetime import date
from pathlib import Path

import polars as pl
import yaml

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("Raiz del repositorio:", ROOT)


def leer_csv(ruta):
    try:
        return pl.read_csv(ruta)
    except FileNotFoundError:
        print(f"No existe {ruta}. Ver la seccion 'Como reproducir'.")
    except Exception as error:
        print(f"No se pudo leer {ruta}: {error}")


def leer_yaml(ruta):
    with open(ruta, encoding="utf-8") as archivo:
        return yaml.safe_load(archivo)

## 1. Objetivo y criterio de selección

Pronosticar el nivel diario del Orinoco en **Ciudad Bolívar** y **Palúa** a 1 a 14 días,
reportando en particular los **días 7 y 14** (instrucción del profesor). Este cuaderno cuenta
cómo se construyó el modelo **TCN**: qué se decidió, por qué y qué se descartó.

**Criterio de selección** (fijado antes de ver resultados):

1. `rel_mae < 1` en **cada** estación, donde `rel_mae` = MAE del modelo / MAE de la persistencia
   (Naive: repetir el último nivel observado) en los mismos cortes.
2. Entre los que cumplen, el menor MAE medio entre las dos estaciones.

**Por qué `rel_mae` y no MASE.** Resuelve el problema de comparar contra una referencia justa.
Por dentro, divide dos errores calculados sobre las mismas ventanas. Se descartó el MASE con
escala del historial de entrenamiento: esa escala puede dar valores mayores que 1 a la propia
persistencia si el período evaluado fue más difícil que el de entrenamiento, y entonces el
umbral "menor que 1" deja de significar algo. `rel_mae` sería la decisión equivocada si la
persistencia fuera casi perfecta (MAE cercano a cero), porque el cociente se vuelve inestable.

## 2. Datos

**Fuente:** `data/raw/DATOS RIO ORINOCO.xlsx`, hoja `Entrenar` (niveles diarios de Ayacucho,
Caicara, Ciudad Bolívar y Palúa), usando **solo 2009-01-01 a 2024-12-31** (período válido
según el profesor): 5 844 días por estación. Los archivos de `data/raw/` no se modifican.

**Qué ve el modelo.** El frame que reciben todos los runners es el parquet de
`TimeSeriesDataset` (`data/interim/orinoco_dataset.parquet`) limpiado con `clean_history`
([`features/dataset.py`](../src/orinoco_forecast/features/dataset.py)). Trae las **cuatro
estaciones como series** (`unique_id`), el nivel `y`, lluvia de NASA POWER (`rain_*`), ONI y
columnas de rezagos ya calculadas. El TCN declara las covariables como históricas (`hist_exog_list`) y `neuralforecast` las corta a los `input_size` días que terminan en cada corte. `drop_covariates: [rain_, oni]` le oculta la lluvia y el ONI.

### Limpieza (`clean_history`) y por qué

| Paso | Qué resuelve | Cómo funciona | Alternativa descartada | Cuándo sería un error |
|---|---|---|---|---|
| Recortar a 2009-2024 | Datos fuera del período no son válidos | Filtra `ds` con `model_frame.start/end` de `configs/data.yaml` | Usar la serie desde 1974: más historia, pero el profesor la declaró no válida | Si se confirma que el período anterior sí es válido |
| Anular rachas planas (`suspect_flat`, 10 días o más iguales) | Un sensor detenido repite el mismo valor y enseñaría persistencia falsa | `label_quality` etiqueta y `clean_series` las pone en nulo | Dejarlas: contamina el aprendizaje | Si una racha plana fuera real (río en estiaje extremo) |
| Interpolar huecos de hasta 31 días | Los modelos no admiten nulos | Interpolación lineal por fecha (`max_gap_days: 31`); marca `imputed = True` | Rellenar con la media o con el último valor: distorsiona la dinámica | Si el hueco cubre una crecida: la línea recta la borra |
| Descartar los primeros 60 días de 2009 | Las columnas de rezagos (hasta `oni_lag_60`) aún no existen | `drop_nulls()` tras limpiar; la serie efectiva empieza el 2009-03-02 | Rellenar rezagos con ceros: inventa datos | Nunca es un error mientras no se invente nada |

En 2009-2024: 3 valores `missing` en Palúa (29 de febrero de 2012, 2016 y 2020), rachas planas
de Ayacucho (2019-03-07 a 2019-03-18) y Ciudad Bolívar (2020-03-24 a 2020-04-11), y un
`suspect_jump` de Palúa (2021-10-29) que se etiqueta pero no se modifica: **34 filas
interpoladas**. La última interpolada (2020-04-11) es anterior a todos los cortes de
validación, así que la interpolación no usa datos posteriores a un corte.

In [ ]:
try:
    from orinoco_forecast.evaluation.execution import load_history

    historial = load_history()
    resumen = (
        historial.group_by("unique_id")
        .agg(
            pl.len().alias("dias"),
            pl.col("ds").min().alias("desde"),
            pl.col("ds").max().alias("hasta"),
            pl.col("imputed").sum().alias("dias_imputados"),
        )
        .sort("unique_id")
    )
    print(resumen)
    print("Columnas que reciben los runners:", len(historial.columns))
    print(historial.columns)
except FileNotFoundError:
    print(
        "Falta data/interim/orinoco_dataset.parquet. Generalo con:\n"
        "  uv run python scripts/run_model.py booster --write-dataset"
    )
except Exception as error:
    print("No se pudo cargar el historial:", error)

## 3. Partición de los datos

| Bloque | Qué es |
|---|---|
| Entrenamiento | Todo `ds <= cutoff`; **crece** con cada corte. Cada corte reajusta el modelo. |
| Validación | Los 14 días siguientes a cada corte: **24 cortes** de 2021-01-26 a 2022-12-17, uno cada 30 días. |
| Test final | `ds >= 2023-01-01`. **Nunca se usó** para decidir nada. |

**Qué problema resuelve.** Una serie de tiempo no se puede barajar ni repartir al azar: el
modelo no debe ver el futuro de lo que predice. **Cómo funciona:** `walk_forward_splits`
([`data/splits.py`](../src/orinoco_forecast/data/splits.py)) genera los cortes hacia atrás
desde `test_start - (horizonte + 1)` y falla si una ventana invade el test. **Alternativa
descartada:** un reparto fijo tipo 70/15/15, que mide un solo período y no dice si el modelo
es consistente; con 24 cortes se mide la consistencia. **Sería un error** si hubiera pocos
datos (aquí 24 ventanas de 14 días son una muestra razonable, no grande).

El test se evalúa **una sola vez**, con el modelo que elija el equipo. Si se mira antes para
decidir, la predicción deja de ser genuina.

In [ ]:
try:
    from orinoco_forecast.data.splits import walk_forward_splits

    cfg = leer_yaml("configs/backtest.yaml")
    test_start = date.fromisoformat(str(cfg["test_start"]))
    fechas = historial.filter(pl.col("unique_id") == "ciudad_bolivar").sort("ds")["ds"].to_list()
    cortes = walk_forward_splits(
        fechas,
        test_start=test_start,
        horizon=cfg["horizon"],
        step_size=cfg["step_size"],
        n_windows=cfg["n_windows"],
    )
    print(len(cortes), "cortes:", cortes[0].cutoff, "a", cortes[-1].cutoff)
    print("La ultima validacion termina el", cortes[-1].validation_end, "| el test empieza el", test_start)
    assert cortes[-1].validation_end < test_start
    solapadas = [b.validation_start <= a.validation_end for a, b in zip(cortes, cortes[1:], strict=False)]
    print("Ventanas de validacion solapadas:", any(solapadas))
except NameError:
    print("Ejecuta antes la celda de datos (falta el historial).")
except Exception as error:
    print("No se pudo calcular la particion:", error)

## 4. Diseño del modelo

Código: [`models/neural/cnn.py`](../src/orinoco_forecast/models/neural/cnn.py) (`TCNRunner`,
sobre el `TCN` de `neuralforecast` 3.3.0) y configuración
[`configs/models/cnn.yaml`](../configs/models/cnn.yaml).

Una **convolución** aplica el mismo filtro (un conjunto pequeño de pesos) a cada tramo de la
serie. En una **convolución causal** la salida de un día solo depende de ese día y de los
anteriores. Una **convolución dilatada** salta días: con dilatación 4 el filtro mira `t`, `t-4`,
`t-8`... Apilando capas con dilataciones 1, 2, 4, 8 y 16 la red ve muy atrás con pocas capas:

```text
campo receptivo = 1 + (kernel_size - 1) x suma(dilations) = 1 + 1 x 31 = 32 días
```

| Decisión | Qué problema resuelve | Cómo funciona por dentro | Alternativa descartada y por qué | Cuándo sería un error |
|---|---|---|---|---|
| **Convolución causal dilatada** | Ver 32 días de historia sin una red profunda ni recurrente | Capas `Conv1d` con relleno a la izquierda y dilataciones 1-2-4-8-16 | LSTM pura: secuencial, más lenta de entrenar y con memoria que se desvanece; es el contraste del tercer cuaderno | Si el retardo físico del río superara el campo receptivo |
| **`input_size: 112`** (28, 56 y 112 buscados; debe ser >= campo receptivo) | Dar al modelo unas 3 veces el campo receptivo para que la capa final vea datos reales | `neuralforecast` corta la ventana de entrada a los 112 días que terminan en el corte | Ventana de 28 días: menor que el campo receptivo, parte de la red vería ceros (el runner la rechaza) | Si la memoria útil del río fuera más corta: la ventana larga añadiría ruido |
| **Objetivo en diferencias** (`target_transform: difference`) | La primera versión predecía el nivel y perdía contra la persistencia en el día 1: el nivel de mañana es casi el de hoy y la red tenía que reproducirlo exacto | Predice `y_t - y_(t-1)`, agrega el nivel actual como covariable `level_y` y reconstruye sumando los cambios al nivel del corte | Predecir niveles: más simple pero obliga a la red a acertar el valor exacto | Si la serie no tuviera inercia (no es el caso) |
| **Pérdida MAE** | Optimizar lo mismo que la métrica de selección | `loss: MAE` de `neuralforecast` | MSE: castiga más las crecidas y deja de coincidir con el MAE del criterio | Si el objetivo fuera acertar las crecidas extremas |
| **Escalado robusto** (mediana y rango intercuartil) | Que las crecidas no distorsionen la escala | `scaler_type: robust`, calculado por ventana con su parte de entrada | Estandarizar con media y desviación: una crecida infla la desviación | Si las crecidas fueran lo único que importa |
| **3 semillas** | Una red parte de pesos aleatorios y dos entrenamientos dan resultados distintos | Se entrenan 3 redes sobre las mismas ventanas; se reporta media y desviación; las predicciones no se promedian | Una sola semilla: no distingue una mejora real de la suerte | Si el costo de cómputo fuera prohibitivo |
| **Parada temprana** sobre los últimos 365 días de cada corte | Evitar sobreajuste | Paciencia de 5 controles cada 50 pasos; esos días no actualizan los pesos | Entrenar un número fijo de pasos: sobreajusta o subajusta según el corte | Si hubiera muy pocos datos |
| **`drop_covariates: [rain_, oni]`** | Decidir qué entradas ve el modelo con validación | Oculta las columnas cuyo nombre empieza con esos prefijos | Pasarle todo (lo que hace por defecto el flujo de runners): se sobreajusta (sección 6) | Si la lluvia aportara información que los niveles no tienen |

### Sin fuga con las covariables históricas

En `neuralforecast` las covariables históricas (`hist_exog_list`) se cortan a los `input_size`
días que terminan en cada corte. Hay una prueba empírica
([`tests/test_cnn.py`](../tests/test_cnn.py), [`tests/test_runner_tcn.py`](../tests/test_runner_tcn.py)):
alterar los valores posteriores al corte deja los pronósticos idénticos.

In [ ]:
try:
    from orinoco_forecast.models.neural.cnn import receptive_field

    for kernel, dilaciones in [(2, [1, 2, 4, 8, 16]), (3, [1, 2, 4, 8, 16]), (2, [1, 2, 4])]:
        campo = receptive_field(kernel, dilaciones)
        print(f"kernel={kernel} dilaciones={dilaciones} -> campo receptivo {campo} dias")
    assert receptive_field(2, [1, 2, 4, 8, 16]) == 32
    print("Con input_size=28 (< 32) el runner rechaza la configuracion; con 112 la capa final ve datos reales.")
except Exception as error:
    print("No se pudo importar el TCN:", error)

In [ ]:
# Objetivo en diferencias con una serie de juguete (sin modelo): persistencia = cambio cero
niveles = pl.DataFrame({"dia": list(range(6)), "y": [5.0, 5.2, 5.1, 5.6, 6.0, 6.1]})
cambios = niveles.with_columns(cambio=pl.col("y").diff())
print(cambios)

nivel_en_el_corte = niveles["y"][3]  # el corte es el dia 3
cambios_predichos = [0.0, 0.0]       # "no cambia": es exactamente la persistencia
reconstruido = [nivel_en_el_corte + sum(cambios_predichos[: i + 1]) for i in range(len(cambios_predichos))]
print("Cambio predicho cero ->", reconstruido, "(igual al ultimo nivel observado:", nivel_en_el_corte, ")")

In [ ]:
try:
    cnn = leer_yaml("configs/models/cnn.yaml")
    for clave in ["target_transform", "input_size", "kernel_size", "dilations", "encoder_hidden_size",
                  "decoder_hidden_size", "decoder_layers", "loss", "learning_rate", "max_steps",
                  "scaler_type", "val_size", "early_stop_patience_steps", "drop_covariates", "seeds"]:
        print(f"{clave}: {cnn[clave]}")
except Exception as error:
    print("No se pudo leer configs/models/cnn.yaml:", error)

## 5. Ajuste de hiperparámetros con Optuna

Las decisiones del método son las mismas que en el cuaderno de XGBoost: **TPE**, **20 pruebas
completas** por modelo, objetivo **`rel_mae` medio**, **12 ventanas de ajuste en 2020** (cortes
del 2020-05-04 al 2021-01-12, cada 23 días) para no usar los datos de validación. Cambia lo
específico del TCN:

| Decisión | Qué problema resuelve | Cómo funciona | Alternativa descartada | Cuándo sería un error |
|---|---|---|---|---|
| **Una semilla por prueba** | Acotar el costo del ajuste | Cada prueba entrena una red; la validación final usa 3 | 3 semillas por prueba: triplica el tiempo | Si el ruido entre semillas fuera mayor que las diferencias entre pruebas |
| **Podar sin entrenar** si el campo receptivo supera `input_size` | No gastar tiempo en configuraciones inválidas | Devuelve `TrialPruned` antes de entrenar (7 pruebas podadas, menos de 0,2 s) | Entrenarlas y dejar que fallen | Nunca |
| **Buscar `target_transform`** (nivel o diferencia) | Confirmar con datos que las diferencias ganan | Las dos opciones entran al espacio | Fijarlo a mano | Si se confía sin evidencia: aquí las mejores pruebas completas usan diferencias |

Espacio buscado: `input_size` 28, 56 o 112; `kernel_size` 2 o 3; dilataciones (`1-16`, `1-32`);
`encoder_hidden_size` 32, 64 o 128; `learning_rate` 1e-4 a 3e-3; `max_steps` 500 o 1000.
Como el mejor puntaje (0,606) se separa mucho del resto de las completas (0,723 la segunda), el
ajuste del TCN es menos estable que el de XGBoost. Un solo hiperparámetro cambió de verdad en el
reajuste con datos 2009-2024: la tasa de aprendizaje (0,002656 a 0,002072).

In [ ]:
pruebas = leer_csv("reports/tuning/tcn_trials.csv")
if pruebas is not None:
    print(pruebas["state"].value_counts())
    completas = pruebas.filter(pl.col("state") == "COMPLETE").sort("value")
    columnas = [c for c in ["number", "value", "rel_mae_ciudad_bolivar", "rel_mae_palua",
                            "params_target_transform", "params_input_size", "params_encoder_hidden_size",
                            "params_learning_rate", "duration_s"] if c in completas.columns]
    with pl.Config(tbl_cols=-1, tbl_width_chars=220, float_precision=4):
        print(completas.select(columnas).head(5))
    mejor = completas.row(0, named=True)
    print("Mejor prueba:", mejor["number"], "| rel_mae medio", round(mejor["value"], 4))
    print("Segunda mejor:", round(completas["value"][1], 4))

## 6. Validación en 24 cortes

Promedios de los 24 cortes (2021-01-26 a 2022-12-17), contra Naive, MAE en metros; TCN: media
de 3 semillas (± desviación entre semillas). **Procedencia:** `run_model.py tcn --against naive
--folds` ejecutado en una GPU de Colab (T4), con `configs/models/cnn.yaml` tal como está.

| Estación | Horizonte | Naive | TCN MAE ± desv. | `rel_mae` ± desv. |
|---|---|---|---|---|
| Ciudad Bolívar | día 7 | 0,526 | 0,274 ± 0,008 | 0,52 ± 0,02 |
| Ciudad Bolívar | día 14 | 1,064 | 0,502 ± 0,046 | 0,47 ± 0,04 |
| Ciudad Bolívar | días 1 a 14 | 0,570 | 0,292 ± 0,013 | 0,51 ± 0,02 |
| Palúa | día 7 | 0,435 | 0,360 ± 0,018 | 0,83 ± 0,04 |
| Palúa | día 14 | 0,891 | 0,560 ± 0,015 | 0,63 ± 0,02 |
| Palúa | días 1 a 14 | 0,488 | 0,342 ± 0,013 | 0,70 ± 0,03 |

**Cumple el criterio** (`rel_mae < 1`) en las dos estaciones y los tres horizontes. Frente al
XGBoost (MAE de los días 1 a 14): en Ciudad Bolívar queda algo por detrás (0,292 contra 0,278)
y en **Palúa empata** (0,342 contra 0,341); la diferencia media (0,008 m) es menor que la
variación entre semillas del TCN, así que no es concluyente. **El punto débil es Palúa a 7
días** (`rel_mae` 0,83): el test de signos da p entre 0,54 y 1,00 según la semilla, sin
evidencia de que gane a Naive. En Ciudad Bolívar a 7 y 14 días la evidencia es clara (p ≤ 0,007).

### Comparación con los otros modelos

Hardware: XGBoost y ARIMAX en CPU, TCN en una GPU de Colab y CNN-LSTM en ambos. Cada celda es
`MAE / rel_mae`. Todo se midió con `run_model.py <modelo> --against naive --folds`.

| Estación | Horizonte | Naive | XGBoost | TCN (GPU) | CNN-LSTM (GPU) | ARIMAX |
|---|---|---|---|---|---|---|
| Ciudad Bolívar | día 7 | 0,526 | 0,279 / 0,53 | 0,274 / 0,52 | 0,307 / 0,58 | 0,284 / 0,54 |
| Ciudad Bolívar | día 14 | 1,064 | 0,444 / 0,42 | 0,502 / 0,47 | 0,620 / 0,58 | 0,576 / 0,54 |
| Ciudad Bolívar | días 1 a 14 | 0,570 | 0,278 / 0,49 | 0,292 / 0,51 | 0,337 / 0,59 | 0,315 / 0,55 |
| Palúa | día 7 | 0,435 | 0,360 / 0,83 | 0,360 / 0,83 | 0,378 / 0,87 | 0,377 / 0,87 |
| Palúa | día 14 | 0,891 | 0,548 / 0,62 | 0,560 / 0,63 | 0,613 / 0,69 | 0,662 / 0,74 |
| Palúa | días 1 a 14 | 0,488 | 0,341 / 0,70 | 0,342 / 0,70 | 0,364 / 0,75 | 0,387 / 0,79 |

MAE medio entre estaciones (días 1 a 14): XGBoost 0,309 m, **TCN 0,317 m**, CNN-LSTM
0,350 m, ARIMAX 0,351 m. El CNN-LSTM no mejora al TCN y cuesta más de 2 horas en CPU.

### Pregunta abierta: hardware

La medición anterior del TCN, hecha en CPU con el script auxiliar ya retirado, dio 0,294 en
Ciudad Bolívar (casi igual que ahora, 0,292) pero **0,383 en Palúa** frente a 0,342 en GPU. La
diferencia es mayor que la variación entre semillas (0,013) y no sabemos si viene del hardware o
del script. El TCN **no se volvió a ejecutar en CPU con `--folds`**. En el CNN-LSTM, el único
modelo medido en los dos, CPU y GPU coinciden en MAE (diferencias de 0,000 a 0,009), lo que
sugiere que el hardware no explica por sí solo una diferencia de 0,04, pero no lo resuelve.

### La lluvia y el ONI perjudican al TCN

Experimento sobre 6 cortes (uno cada 120 días, del 2021-04-26 al 2022-12-17; una muestra más
difícil que los 24 cortes). `rel_mae` de los días 1 a 14, media de 3 semillas:

| Entrada del TCN | Ciudad Bolívar | Palúa |
|---|---|---|
| Todas las columnas (con lluvia y ONI) | 0,98 | 1,12 |
| **Solo niveles** (la configuración final) | **0,71** | **0,80** |
| Sin covariables | 0,94 | 1,04 |

Con lluvia y ONI la red se sobreajusta y llega a perder contra la persistencia. Los niveles
rezagados de las estaciones aguas arriba sí aportan: sin ninguna covariable también empeora.
Por eso `cnn.yaml` declara `drop_covariates: [rain_, oni]`.

*Procedencia:* este experimento (a diferencia de la tabla de arriba) se hizo con un script
auxiliar equivalente, ya retirado, y no se volvió a ejecutar.


In [ ]:
def comparacion(runner):
    ruta = ROOT / f"data/predictions/{runner}/backtest/naive/comparison.parquet"
    try:
        tabla = pl.read_parquet(ruta)
    except FileNotFoundError:
        print(f"No existe {ruta}.\nGeneralo con: uv run python scripts/run_model.py {runner} --against naive --folds")
        return None
    except Exception as error:
        print(f"No se pudo leer {ruta}: {error}")
        return None
    try:
        columnas = ["unique_id", "model", "horizon", "mean_model", "mean_baseline",
                    "mean_improvement_pct", "win_rate", "sign_test_p"]
        return (
            tabla.filter((pl.col("metric") == "mae") & pl.col("horizon").cast(pl.String).is_in(["7", "14", "all"]))
            .select(columnas)
            .sort("unique_id", "model", "horizon")
        )
    except Exception as error:
        print("El archivo existe pero no tiene el formato esperado:", error)
        return tabla


resultado = comparacion("tcn")
if resultado is not None:
    with pl.Config(tbl_cols=-1, tbl_width_chars=200, float_precision=4):
        print(resultado)

## 7. Limitaciones y siguientes pasos

- **Reajuste por corte vs Optuna.** `--folds` reentrena en cada corte; Optuna eligió los
  hiperparámetros con un ajuste único y 12 ventanas de 2020. Los parámetros siguen siendo
  válidos, pero la medición no es idéntica.
- **Sin Fourier.** El parquet no trae `sin_*`/`cos_*` (`calendar_terms: 0`), así que el modelo
  no ve el ciclo anual de forma explícita. No se midió cuánto cambiaría.
- **Entrenan con las cuatro estaciones.** `execute` pasa las cuatro series y filtra la
  predicción a las dos objetivo. No se probó entrenar solo con las dos objetivo.
- **NSE negativo.** Cada ventana dura 14 días y su varianza interna es pequeña, así que el
  NSE castiga cualquier error (también a Naive). Se interpreta `rel_mae`.
- **Una validación de 24 cortes es una muestra moderada.** Los resultados se leen junto con el
  test de signos, no solo con el promedio.
- **Test final sin tocar.** Estos resultados son de validación y pueden ser algo optimistas.
- **Sin datos de la represa de Guri** (regula el Caroní, cerca de Palúa).
- **Hardware mezclado.** Las cifras del TCN salen de una GPU de Colab, que no es determinista,
  y su medición antigua en CPU difiere en Palúa (0,383 contra 0,342); sin resolver.
- **Una semilla por prueba en Optuna** y **mejor prueba aislada**: el ajuste es menos estable que el del booster.
- **El experimento de covariables usa 6 cortes**, una muestra más ruidosa y más difícil.
- **La importancia de las covariables** no se midió; solo se compararon los tres conjuntos de entradas.

**Siguientes pasos:** repetir el TCN en CPU con `--folds` para cerrar la pregunta de hardware, probar
Fourier y acordar con el equipo el protocolo de la evaluación final (el TCN se conserva como referencia).

## 8. Cómo reproducir

```bash
uv sync
# Genera el parquet (una vez)
uv run python scripts/run_model.py booster --write-dataset
# Un corte contra Naive
uv run python scripts/run_model.py tcn --against naive
# 24 cortes (salidas en data/predictions/tcn/backtest/naive/)
uv run python scripts/run_model.py tcn --against naive --folds
```

Para usar GPU en Colab: [`colab_folds.ipynb`](colab_folds.ipynb) (cambia la variable
`MODEL`). Los resultados en GPU pueden variar un poco respecto a CPU porque el runner no activa
las opciones de determinismo de cuDNN.